# RMSNorm — Solution

Reference implementation for `02_rmsnorm_exercise.ipynb`.

## Setup

In [1]:
import torch 
import torch.nn as nn

## Solution 1 — `LayerNorm`

In [19]:
class LayerNorm(nn.Module):
    def __init__(self,hidden_dim):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(hidden_dim))
        self.beta = nn.Parameter(torch.zeros(hidden_dim))

    def forward(self,x):
        # x -> bs,seq_len,hidden_dim
        mean = x.mean(axis=-1,keepdims=True)
        var = x.var(axis=-1,keepdims=True,unbiased=False)

        x = (x-mean) / torch.sqrt(var+1e-7)

        x = self.gamma * x + self.beta

        return x

In [20]:
k = LayerNorm(hidden_dim=8)


In [21]:
k(torch.randn((2,4,8)))

tensor([[[ 0.1787, -0.6034, -1.8105,  0.2135, -0.3150, -0.4697,  1.5094,
           1.2970],
         [-0.6066,  0.9520,  0.1720, -0.6711,  0.7877, -2.0348,  0.1984,
           1.2022],
         [ 1.9319,  0.2588,  0.7421, -1.4041, -1.1459, -0.1879, -0.4920,
           0.2970],
         [ 1.0852, -1.2482,  0.3658,  0.0663, -0.6195,  1.7523, -0.1138,
          -1.2881]],

        [[ 0.6901, -1.4716,  1.1460,  0.5357, -0.4210,  0.1529, -1.6117,
           0.9796],
         [ 1.4012, -1.1187,  1.7021, -0.8961, -0.0312, -0.0353, -1.0403,
           0.0182],
         [-0.7370,  0.6947,  1.6246, -0.7579, -1.7971,  0.4861,  0.5399,
          -0.0533],
         [-1.2273,  1.5043,  0.1185,  0.7202,  0.2177, -1.7742, -0.2299,
           0.6708]]], grad_fn=<AddBackward0>)

## Solution 2 — `RMSNorm`

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self,hidden_dim):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(hidden_dim))

    def forward(self,x):
        # x -> bs,seq_len,hidden_dim
        mean_squared = torch.mean(x**2,axis=-1,keepdim=True)

        x = x / torch.sqrt(mean_squared+1e-7)

        x = self.gamma * x

        return x

**Why is mean subtraction safe to drop?** A constant shift on the input vector is a linear operation. The next layer's weight matrix and bias absorb it trivially — it's already in the function class. So mean subtraction is fixing a non-problem in terms of expressiveness. The MAGNITUDE growing across layers, on the other hand, breaks training catastrophically (1.1^60 ≈ 304). RMSNorm keeps the magnitude fix, drops the centering.

**Why no bias parameter (beta)?** For the same reason: a bias is a constant additive shift. Anything beta could do, the next layer's weights can do. Removing it saves params and matches the modern "no biases anywhere" convention.

## Run the tests

In [26]:
from tests import run_norm_tests
run_norm_tests(LayerNorm, RMSNorm)

Running Normalization...
  ✓ LayerNorm shape correct
  ✓ LayerNorm output: zero mean
  ✓ LayerNorm output: unit variance
  ✓ RMSNorm shape correct
  ✓ RMSNorm output: unit RMS
  ✓ RMSNorm has fewer params (no beta)

  All 6 checks passed ✓



True